In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import linregress

from armored.models import *
from armored.preprocessing import *

import itertools

from tqdm import tqdm

import matplotlib.pyplot as plt

In [ ]:
# data with initial and end point measurements
df_exp0 = pd.read_csv("data/exp0/exp0_metabolites.csv")
df_exp1 = pd.read_csv("data/exp1/exp1_metabolites.csv")
df_exp2 = pd.read_csv("data/exp2/exp2_metabolites.csv")
df = pd.concat((df_exp0, df_exp1, df_exp2))

In [ ]:
species = ['ACabs', 'BAabs', 'BHabs', 'BLabs', 'BUabs', 'CAabs', 'CCabs', 'CHabs',
           'DFabs', 'ELabs', 'ERabs', 'FPabs', 'PCabs', 'PJabs', 'RIabs']
metabolites = ['pH', 'Lactate', 'Butyrate', 'Acetate']
controls = ['AcGum', 'ArGal', 'Inulin', 'Pectin', 'Starch', 'Xylan']

# concatenate all observed and all system variables 
observed = np.concatenate((np.array(species), np.array(metabolites)))
system_variables = np.concatenate((np.array(species), np.array(metabolites), np.array(controls)))
system_variables

In [ ]:
# scale data 
scaler = MinMaxScaler(observed, system_variables)
scaler.fit(df)
df_scaled = scaler.transform(df.copy())

# format data into matrix [n_samples, n_timepoints, dt+n_outputs+n_controls]
data = format_data(df.copy(), species, metabolites, controls, observed=observed)
data_scaled = format_data(df_scaled, species, metabolites, controls, observed=observed)

# instantiate model
brnn = miRNN(n_species=len(species), n_metabolites=len(metabolites), n_controls=len(controls), n_hidden=16)
# fit model
brnn.fit(data_scaled, evd_tol=1e-3)

In [ ]:
# full factorial of both species and controls 
n_species = len(species)
n_fibers  = len(controls)

# create matrix of all possible communities
Slist = [np.reshape(np.array(i), (1, n_species)) for i in itertools.product([0, 1], repeat = n_species)]
# remove all zeros community
S = np.array(np.concatenate(Slist)[1:], float)

# create matrix of all possible fiber permutations
def generate_permutations(n, k, prefix=[]):
    if k == 0:
        if n == 0:
            yield prefix
        return
    
    for i in range(n + 1):
        if i <= n:
            yield from generate_permutations(n - i, k - 1, prefix + [i])

def generate_all_permutations(n):
    return np.array(list(generate_permutations(n, n)))/n
F = generate_all_permutations(n_fibers)

# matrix of species and fiber indeces 
M = np.array(np.zeros([S.shape[0]*F.shape[0], 2]), int)
k = 0 
for i in range(S.shape[0]):
    for j in range(F.shape[0]):
        M[k, 0] = int(i)
        M[k, 1] = int(j)
        k += 1

# function to pull sample data 
def gen_exp_cond(k):
    s_ind, f_ind = M[k]
    return S[s_ind], F[f_ind]

# function to generate informative name of experimental condition
def gen_exp_name(Si, Fi):
    exp_name = ""
    for i,si in enumerate(Si):
        if si > 0:
            exp_name += species[i].split("abs")[0] + "-"
    for i,fi in enumerate(Fi):
        if fi > 0:
            exp_name += str(int(fi*6)) + controls[i] + "-"
    return exp_name[:-1]

In [ ]:
def format_design_data(S, F, t_eval, scaler, batch_size=512):

    # data is a list of tuples (T, X, U, Y, names) where each tuple corresponds to a batch
    data = []
    
    # total number of samples
    n_samples = S.shape[0] * F.shape[0]
    k = 0 
    
    # number of evaluation times
    n_eval = len(t_eval)

    # divide data into batches
    for batch_inds in tqdm(np.array_split(np.arange(n_samples), np.ceil(n_samples / batch_size))):

        # initialize data matrices 
        T = np.empty([len(batch_inds), n_eval])
        T[:] = t_eval
        X = np.empty([len(batch_inds), S.shape[-1]+len(metabolites)])
        X[:] = np.nan
        U = np.empty([len(batch_inds), n_eval, F.shape[-1]])
        U[:] = np.nan

        # keep track of experiment names
        names = []
        for i, batch_ind in enumerate(batch_inds):

            # pull sample data 
            Si, Fi = gen_exp_cond(k)

            # keep track of experiment names
            names.append(gen_exp_name(Si, Fi))

            # store initial condition data
            X[i, :len(Si)] = .000667 * Si
            
            # pH and metabolites
            X[i, len(Si):] = np.array([6.7, 0., 0., 0.])
            
            # scale observed variables 
            X[i] = (X[i] - scaler.scale_dict_obs["0 min"]) / scaler.scale_dict_obs["0 max"]
            
            # store controls (already scaled)
            U[i] =  Fi 
            
            # update counter
            k += 1

        data.append((T, X, U, names))

    return data

In [ ]:
# format and scale data based on training data
design_data = format_design_data(S, F, np.array([0, 1, 2, 3]), scaler, batch_size=1024)

In [ ]:
# log that ignores zeros
def zlog(x):
    return jnp.where(x > 0, jnp.log(x), 0.)

# shannon diversity
def shannon(y):
    y_rel = y / jnp.sum(y)
    return jnp.nansum(-zlog(y_rel)*y_rel)

# determine best previously observed values of objectives
b_max = 0
d_max = 0 
v_max = 0
for t, x, u, y, n in data:
    for yi in y:
        b_max = np.max([b_max, np.nan_to_num(yi[-1, -2])])
        d_max = np.max([d_max, shannon(yi[-1, :len(species)])])
        
        species_stdv = jnp.nanstd(yi[-2:, :len(species)], 0)
        if sum(species_stdv>0) > 0:
            v_max = np.max([v_max, jnp.where(species_stdv>0, species_stdv, 0).mean()])

# define objective 
def objective(y_scaled):
    # y is predicted trajectory [n_time, n_species + n_metabolites]
    y = jnp.zeros_like(y_scaled)
    
    # reverse scaling on y 
    for t, (eval_time, y_t) in enumerate(zip(scaler.eval_times, y_scaled)):
        # x = x.at[idx].set(y)
        y = y.at[t].set(y_t*(scaler.scale_dict_obs[f"{eval_time} max"] - scaler.scale_dict_obs[f"{eval_time} min"]) 
                        + scaler.scale_dict_obs[f"{eval_time} min"])
    
    # endpoint shannon diversity
    diversity = shannon(y[-1, :len(species)]) / d_max
    
    # variance in species abundances in last two passages
    species_stdv = jnp.std(y[-2:, :len(species)], 0)
    instability  = jnp.where(species_stdv>0, species_stdv, 0).mean() / v_max
    
    # endpoint butyrate production 
    butyrate = y[-1, -2] / b_max 
    
    return diversity - instability + butyrate

In [ ]:
# function to predict metabolites and variance
def predict_point_params(self, X, U, params):
    # make point predictions
    preds = nn.relu(self.forward_batch(params, X, U))

    # include known initial conditions
    preds = np.concatenate((np.expand_dims(X, 1), preds), 1)

    return preds

# sample from multivariate Gaussian
def sample(mu, cov):        
    
    # cholesky factorization of covariance
    L = np.linalg.cholesky(cov)
    
    # standard normal random vector
    z = np.random.randn(len(mu))
    
    # transform so that sample has E[s] = mu, cov[s] = cov
    return mu + np.dot(z, L.T)

In [ ]:
# return indices of optimal samples using Thompson sampling
def search_Thompson(self, data, objective, n_design, max_repeats=100):

    # compute profit function (f: R^[n_t, n_o] -> R) in batches
    objective_batch = jit(vmap(objective))

    # init copy of parameter covariance
    Ainv_q = jnp.copy(self.Ainv)
    
    # search for new experiments until find N
    best_experiments = []
    repeats = 0
    while len(best_experiments) < n_design and repeats < max_repeats:

        # sample parameter values from posterior 
        params = sample(self.params, Ainv_q)
        
        # evaluate objectives 
        f_P = []
        f_P_max = []
        for T, X, U, exp_names in tqdm(data):

            # number of samples in each dimension
            n_samples = len(T)

            # make predictions on data
            preds = predict_point_params(self, X, U, params)
            f_P_i = objective_batch(preds)

            # append to list of objective evaluations
            f_P.append(f_P_i)
            f_P_max.append(np.max(f_P_i))

        # initialize with sample that maximizes objective
        best_dim = np.argmax(f_P_max)
        best_sample = np.argmax(f_P[best_dim])
        T, X, U, exp_names = data[best_dim]
        best_experiment = exp_names[best_sample]
        if best_experiment not in best_experiments:
            repeats = 0
            best_experiments.append(best_experiment)
            print("Picked experiment {}, with predicted outcome of {:.3f}".format(best_experiments[-1],
                                                                                  f_P[best_dim][best_sample]))
        else:
            repeats += 1

    # if have enough selected experiments, return
    return best_experiments

In [ ]:
# search for best experiments over 3 plates 
designed_experiments = search_Thompson(brnn, design_data, objective, n_design=3*96)

In [ ]:
# print set of selected experiments
designed_experiments